## env: GPU

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn as nn
import torch.optim as optim

import time

import shutil

import torchvision.utils
from torch.utils.data import DataLoader, Subset
from torchvision import models
import torchvision.datasets as dsets
import torchvision.transforms as transforms

import torchattacks
from torchattacks import PGD, FGSM
from torchsummary import summary
from sklearn.model_selection import train_test_split

In [2]:
batch_size = 15

full_trainset = torchvision.datasets.ImageFolder(
    root='./data/GTSRB/Final_Training/Images',
    transform=transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
    ])
)

testset = torchvision.datasets.ImageFolder(
    root='./data/GTSRB/test',
    transform=transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
    ])
)

indices = list(range(len(full_trainset)))
labels = full_trainset.targets

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

trainset = Subset(full_trainset, train_indices)
valset = Subset(full_trainset, val_indices)

train_loader = DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    valset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)

test_loader = DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)

## model LNL

In [14]:
from LNL import LNL_Ti as small
model = small(pretrained=False)
model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)
model = model.cuda()

## Train Locality-iN-Locality

In [15]:
num_epochs = 30

In [16]:
checkpoint_dir = '../checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

Validation function

In [17]:
def validate(model, val_loader, criterion):
    model.eval()

    val_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.cuda()
            labels = labels.cuda()

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    avg_loss = val_loss / len(val_loader)
    accuracy = 100 * correct / total

    return avg_loss, accuracy

In [18]:
def train_model(start_epoch, end_epoch):
    best_acc = 0

    loss = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs,eta_min=0)

    # Nếu tiếp tục train từ checkpoint
    if start_epoch != 0:
        checkpoint_file = os.path.join(checkpoint_file, f"checkpoint_epoch_{start_epoch}.pth")
        checkpoint = torch.load(checkpoint_file, map_location="cuda")
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        best_acc = checkpoint.get("best_acc", 0)
        print(f"Loaded checkpoint from epoch {start_epoch}")

    # Train từ start_epoch đến end_epoch
    for epoch in range(start_epoch, end_epoch):
        model.train()

        train_loss, correct, total = 0.0, 0, 0

        for i, (batch_images, batch_labels) in enumerate(train_loader):
            X = batch_images.cuda()
            Y = batch_labels.cuda()

            pre = model(X)

            cost = loss(pre, Y)
            optimizer.zero_grad()
            cost.backward()

            optimizer.step()

            train_loss += cost.item()
            _, predicted = pre.max(1)
            total += Y.size(0)
            correct += predicted.eq(Y).sum().item()

        # Cập nhật learning rate
        scheduler.step()

        train_loss /= len(train_loader)
        train_acc = 100.0 * correct / total

        val_loss, val_acc = validate(model, val_loader, loss)

        if val_acc > best_acc:
            best_acc = val_acc
            checkpoint_file = os.path.join(
                checkpoint_dir,
                f"checkpoint_epoch_{epoch + 1}.pth"
            )

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_acc": best_acc,
            }, checkpoint_file)

            print(f"Checkpoint saved to: {checkpoint_file}\n")
        test_acc = evaluate_model(model, test_loader)
        print(
            f"Epoch [{epoch+1}/{num_epochs}]\n"
            f"\tVal Loss  : {val_loss:.4f}   | Val Acc  : {val_acc:.2f}%\n"
            f"\tTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%\n"
            f"\tTest Acc  : {test_acc:.2f}%\n"
        )


In [19]:
start_epoch = 0
end_epoch = 40
train_model(start_epoch, end_epoch)

Checkpoint saved to: ../checkpoints\checkpoint_epoch_1.pth

Epoch [1/30]
	Val Loss  : 0.7101   | Val Acc  : 99.85%
	Train Loss: 1.0539 | Train Acc: 88.86%
	Test Acc  : 97.32%

Epoch [2/30]
	Val Loss  : 0.7140   | Val Acc  : 99.32%
	Train Loss: 0.7030 | Train Acc: 99.81%
	Test Acc  : 96.98%

Epoch [3/30]
	Val Loss  : 0.7038   | Val Acc  : 99.74%
	Train Loss: 0.6996 | Train Acc: 99.78%
	Test Acc  : 98.00%

Checkpoint saved to: ../checkpoints\checkpoint_epoch_4.pth

Epoch [4/30]
	Val Loss  : 0.6919   | Val Acc  : 99.92%
	Train Loss: 0.6959 | Train Acc: 99.83%
	Test Acc  : 98.47%

Epoch [5/30]
	Val Loss  : 0.6917   | Val Acc  : 99.89%
	Train Loss: 0.6946 | Train Acc: 99.86%
	Test Acc  : 98.65%

Checkpoint saved to: ../checkpoints\checkpoint_epoch_6.pth

Epoch [6/30]
	Val Loss  : 0.6881   | Val Acc  : 99.95%
	Train Loss: 0.6884 | Train Acc: 99.97%
	Test Acc  : 99.11%

Checkpoint saved to: ../checkpoints\checkpoint_epoch_7.pth

Epoch [7/30]
	Val Loss  : 0.6868   | Val Acc  : 99.96%
	Train Lo

KeyboardInterrupt: 

## Test

In [ ]:
model.eval()
correct = 0
total = 0

for images, labels in test_loader:
    images = images.cuda()
    outputs = model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()

print('Standard accuracy: %.2f %%' % (100 * float(correct) / total))

Test a checkpoint

In [ ]:
checkpoint_path = "../checkpoints/99_33.pth"
torch.save({
                "model_state_dict": model.state_dict(),
            }, checkpoint_path)

test_model = small(pretrained=False)
test_model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)
checkpoint = torch.load(checkpoint_path, map_location="cuda")
test_model.load_state_dict(checkpoint["model_state_dict"])
test_model = test_model.cuda()

test_model.eval()

correct = 0
total = 0

for images, labels in test_loader:
    images = images.cuda()
    outputs = test_model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()
    
print('Standard accuracy: %.2f %%' % (100 * float(correct) / total))

Test all checkpoints in checkpoints dir

In [10]:
def evaluate_model(model, test_loader, device=None):
    """
    Evaluate model on test set
    """
    
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    return accuracy

In [ ]:
import glob
from LNL import LNL_Ti as small

def test_checkpoints(checkpoint_dir, test_loader, num_classes=43, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Lấy tất cả file .pth
    checkpoint_paths = glob.glob(os.path.join(checkpoint_dir, "*.pth"))

    # Sắp xếp theo tên file
    checkpoint_paths = sorted(checkpoint_paths)

    results = {}

    for checkpoint_path in checkpoint_paths:
        checkpoint_name = os.path.basename(checkpoint_path)
        print(f"\nTesting: {checkpoint_name}")

        # Tạo model
        test_model = small(pretrained=False)

        # Thay classification head
        test_model.head = nn.Linear(in_features=192, out_features=num_classes, bias=True)

        # Đưa model lên device
        test_model = test_model.to(device)

        # Load checkpoint
        checkpoint = torch.load(checkpoint_path, map_location=device)

        test_model.load_state_dict(checkpoint["model_state_dict"])

        # Evaluate
        accuracy = evaluate_model(test_model, test_loader, device)

        results[checkpoint_name] = accuracy

        print(f"Accuracy: {accuracy:.2f}%")

    return results


checkpoint_dir = "../checkpoints"
results = test_checkpoints(checkpoint_dir=checkpoint_dir, test_loader=test_loader, num_classes=43)
print(results)

## FGSM attack

In [ ]:
model.eval()

correct = 0
total = 0

atk = FGSM(model, eps=0.01)

for images, labels in test_loader:
    
    images = atk(images, labels).cuda()
    outputs = model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()
    
print('Robust accuracy: %.2f %%' % (100 * float(correct) / total))

## PGD attack

In [ ]:
model.eval()

correct = 0
total = 0

atk = PGD(model, eps=0.01, alpha=2/255, steps=5, random_start=False)

for images, labels in test_loader:
    
    images = atk(images, labels).cuda()
    outputs = model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()
    
print('Robust accuracy: %.2f %%' % (100 * float(correct) / total))

## train LNL-MoEx

In [ ]:
from LNL_MoEx import LNL_MoEx_Ti as small
model = small(pretrained=False)
model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)
model = model.cuda()

In [ ]:
import time
# time.clock_gettime()

In [ ]:
num_epochs = 20
moex_lam = .9
moex_prob = .7

In [ ]:
def train_model_TNT_MoEx(start_epoch, end_epoch):
    loss = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs,eta_min=0)

    # Nếu tiếp tục train từ checkpoint
    if start_epoch != 0:
        checkpoint_path = os.path.join(checkpoint_path, f"checkpoint_epoch_{start_epoch}.pth")
        checkpoint = torch.load(checkpoint_path, map_location="cuda")
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        print(f"Loaded checkpoint from epoch {start_epoch}")

    # Train từ start_epoch đến end_epoch
    for epoch in range(start_epoch, end_epoch):
        total_batch = len(train_loader)
        model.train()

        for i, (input, target) in enumerate(train_loader):
            input = input.cuda()
            target = target.cuda()

            prob = torch.rand(1).item()

            if prob < moex_prob:
                swap_index = torch.randperm(input.size(0), device=input.device)
                with torch.no_grad():
                    target_a = target
                    target_b = target[swap_index]
                output = model(input, swap_index=swap_index, moex_norm='pono', moex_epsilon=1e-5,
                                moex_layer='stem', moex_positive_only=False)
                lam = moex_lam
                cost = loss(output, target_a) * lam + loss(output, target_b) * (1. - lam)
            else:
                # compute output
                output = model(input)
                # if args.prof >= 0: torch.cuda.nvtx.range_pop()
                cost = loss(output, target)

            optimizer.zero_grad()
            cost.backward()
            optimizer.step()

            if (i + 1) % 200 == 0:
                print(
                    'Epoch [%d/%d], Iter [%d/%d], Loss: %.6f'
                    % (epoch + 1, num_epochs, i + 1, total_batch, cost.item())
                )

        # Cập nhật learning rate
        scheduler.step()

        # Lưu checkpoint sau mỗi 10 epoch
        if (epoch + 1) % 10 == 0:
            checkpoint_path = os.path.join(
                checkpoint_dir,
                f"checkpoint_epoch_{epoch + 1}.pth"
            )

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
            }, checkpoint_path)

            print(f"Checkpoint saved to: {checkpoint_path}\n")

In [ ]:
start_epoch = 0
end_epoch = 20
train_model_TNT_MoEx(start_epoch, end_epoch)

## Number of Parameters

In [ ]:
# pip install ptflops

In [ ]:
# pip install --upgrade git+https://github.com/sovrasov/flops-counter.pytorch.git

In [ ]:
# import torch
# from ptflops import get_model_complexity_info

# with torch.cuda.device(0):
#   net = model
#   macs, params = get_model_complexity_info(net, (3, 224, 224), as_strings=True,
#                                            print_per_layer_stat=True, verbose=True)
#   print('{:<30}  {:<8}'.format('Computational complexity: ', macs))
#   print('{:<30}  {:<8}'.format('Number of parameters: ', params))
